# Permutation Entropy — Financial Trading Indicators

**Strategy:** Permutation Entropy (PE) acts as a market-regime filter. When PE signals an orderly market, three directional indicators — MACD, RSI, and Moving Average Crossover — generate buy/sell signals.

| # | Indicator | Role | Output |
|---|-----------|------|--------|
| 1 | Permutation Entropy | Chaos/order filter | 0 / 1 |
| 2 | MACD | Trend direction | -1 / 0 / 1 |
| 3 | RSI | Momentum / mean reversion | -1 / 0 / 1 |
| 4 | MA Crossover | Trend confirmation | -1 / 0 / 1 |

**Tickers:** AAPL, MSFT, JPM + SPY (index benchmark)  
**Analysis period:** 2022-01-01 to 2026-04-12

In [ ]:
import requests
import pandas as pd
import numpy as np
import csv
import math
import time
import os
from datetime import datetime
from scipy import stats
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# ---- Configuration ----------------------------------------------------------
API_KEY    = 'ZKMMTO1ATDBLXH2K'
TICKERS    = ['AAPL', 'MSFT', 'JPM']
INDEX      = 'SPY'
START_DATE = '2022-01-01'
END_DATE   = '2026-04-12'

In [ ]:
def download_prices(ticker, start_date, end_date, api_key):
    '''Download daily adjusted close prices from AlphaVantage.
    Returns list of [ticker, date, adjusted_close, log_return].
    '''
    url = ('https://www.alphavantage.co/query'
           '?function=TIME_SERIES_DAILY_ADJUSTED'
           '&symbol=' + ticker +
           '&outputsize=full'
           '&apikey=' + api_key)
    response = requests.get(url)
    data     = response.json()

    ts = data.get('Time Series (Daily)', {})
    if not ts:
        print('  Warning: no data for ' + ticker + '. Keys: ' + str(list(data.keys())))
        return []

    raw = []
    for date_str, vals in ts.items():
        if start_date <= date_str <= end_date:
            raw.append([ticker, date_str, float(vals['5. adjusted close'])])
    raw.sort(key=lambda x: x[1])

    result = []
    for i, rec in enumerate(raw):
        lr = 0.0 if i == 0 else math.log(rec[2] / raw[i - 1][2])
        result.append([rec[0], rec[1], rec[2], lr])
    return result


def download_all_data(tickers, index, start_date, end_date, api_key):
    '''Download + cache all ticker data. Exports one CSV per ticker with ticker and date columns.
    Skips download if CSV already exists (respects AlphaVantage rate limits).
    Returns dict {ticker: [[ticker, date, price, log_return], ...]}.
    '''
    all_data = {}
    for ticker in tickers + [index]:
        csv_file = ticker + '_data.csv'
        if os.path.exists(csv_file):
            print('  ' + ticker + ': loading from ' + csv_file)
            records = []
            with open(csv_file, newline='') as f:
                reader = csv.reader(f)
                next(reader)  # skip header
                for row in reader:
                    records.append([row[0], row[1], float(row[2]), float(row[3])])
        else:
            print('  ' + ticker + ': downloading from AlphaVantage...')
            records = download_prices(ticker, start_date, end_date, api_key)
            with open(csv_file, 'w', newline='') as f:
                writer = csv.writer(f)
                writer.writerow(['ticker', 'date', 'adjusted_close', 'log_return'])
                writer.writerows(records)
            print('    Saved ' + str(len(records)) + ' rows to ' + csv_file)
            time.sleep(15)  # AlphaVantage free tier: 5 req/min
        all_data[ticker] = records
    return all_data

In [ ]:
# Download data on first run; subsequent runs load from cached CSVs
data = download_all_data(TICKERS, INDEX, START_DATE, END_DATE, API_KEY)
for ticker, records in data.items():
    print(ticker + ': ' + str(len(records)) + ' records  (' + records[0][1] + ' to ' + records[-1][1] + ')')

---
## Technical Indicators

Each function takes the full price series and a current date index `i`, returning a signal for that date.

In [ ]:
# ---- Indicator 1: Permutation Entropy (Chaos Filter) -----------------------
#
# Core idea: rank the last `m` prices from lowest (1) to highest (m).
# The sequence of ranks is a "fingerprint". Count how often each fingerprint
# appears across a rolling window. Shannon entropy over those frequencies
# measures how disordered the market is.
#   Low PE  -> few dominant patterns -> orderly market  -> signal = 1
#   High PE -> all patterns equally likely -> chaotic   -> signal = 0

def permutation_entropy_score(prices, m=3, normalize=True):
    '''Compute Permutation Entropy for a price list.
    m         : embedding dimension (ordinal window)
    normalize : if True, divide by log(m!) so result is in [0, 1]
    Returns 0 (fully ordered) to 1 (fully chaotic).
    '''
    n = len(prices)
    if n < m:
        return 1.0  # insufficient data: treat as chaotic

    pattern_counts = {}
    for i in range(n - m + 1):
        pattern = tuple(int(x) for x in np.argsort(prices[i:i + m]))
        pattern_counts[pattern] = pattern_counts.get(pattern, 0) + 1

    total   = sum(pattern_counts.values())
    entropy = 0.0
    for count in pattern_counts.values():
        p = count / total
        entropy -= p * math.log(p)

    if normalize:
        max_h   = math.log(math.factorial(m))
        entropy = entropy / max_h if max_h > 0 else 0.0

    return entropy


def indicator_1_pe(prices_series, date_idx, window=30, m=3, threshold=0.70):
    '''Permutation Entropy regime filter.
    Returns 1 (orderly market: PE < threshold) or 0 (chaotic: step aside).
    Only when this returns 1 should MACD / RSI / MA signals be acted upon.
    '''
    if date_idx < window + m - 1:
        return 0
    recent = prices_series[date_idx - window + 1:date_idx + 1]
    pe     = permutation_entropy_score(recent, m=m)
    return 1 if pe < threshold else 0

In [ ]:
# ---- Indicator 2: MACD ------------------------------------------------------

def _ema(arr, period):
    '''Exponential Moving Average helper.'''
    k   = 2.0 / (period + 1)
    out = np.empty(len(arr))
    out[0] = arr[0]
    for i in range(1, len(arr)):
        out[i] = arr[i] * k + out[i - 1] * (1 - k)
    return out


def indicator_2_macd(prices_series, date_idx, fast=12, slow=26, signal_period=9):
    '''MACD (Moving Average Convergence Divergence).
    Returns  1 if MACD line is above signal line (bullish momentum),
            -1 if below (bearish momentum),
             0 if insufficient data.
    '''
    if date_idx < slow + signal_period:
        return 0
    prices      = np.array(prices_series[:date_idx + 1], dtype=float)
    macd_line   = _ema(prices, fast) - _ema(prices, slow)
    signal_line = _ema(macd_line, signal_period)
    if macd_line[-1] > signal_line[-1]:
        return 1
    elif macd_line[-1] < signal_line[-1]:
        return -1
    return 0

In [ ]:
# ---- Indicator 3: RSI -------------------------------------------------------

def indicator_3_rsi(prices_series, date_idx, period=14, oversold=35, overbought=65):
    '''Relative Strength Index.
    Returns  1 if RSI < oversold  (potential buy: price has fallen too far),
            -1 if RSI > overbought (potential sell: price has risen too far),
             0 otherwise (neutral zone).
    '''
    if date_idx < period + 1:
        return 0
    prices   = np.array(prices_series[max(0, date_idx - period * 3):date_idx + 1], dtype=float)
    deltas   = np.diff(prices)
    gains    = np.where(deltas > 0, deltas,  0.0)
    losses   = np.where(deltas < 0, -deltas, 0.0)
    avg_gain = np.mean(gains[-period:])
    avg_loss = np.mean(losses[-period:])
    rsi = 100.0 if avg_loss == 0 else 100.0 - 100.0 / (1.0 + avg_gain / avg_loss)
    if rsi < oversold:
        return 1
    elif rsi > overbought:
        return -1
    return 0

In [ ]:
# ---- Indicator 4: Moving Average Crossover ----------------------------------

def indicator_4_ma_crossover(prices_series, date_idx, short_window=20, long_window=50):
    '''Simple Moving Average crossover.
    Returns  1 if short MA > long MA (uptrend confirmed),
            -1 if short MA < long MA (downtrend confirmed),
             0 if insufficient data or MAs are flat relative to each other.
    A 0.1% buffer suppresses whipsaw around the crossover point.
    '''
    if date_idx < long_window:
        return 0
    prices   = np.array(prices_series[:date_idx + 1], dtype=float)
    short_ma = np.mean(prices[-short_window:])
    long_ma  = np.mean(prices[-long_window:])
    if short_ma > long_ma * 1.001:
        return 1
    elif short_ma < long_ma * 0.999:
        return -1
    return 0

---
## Trading Strategy

**Recommendation logic:**
- PE = 0 (chaotic) → **no trade** regardless of other signals
- PE = 1 (orderly) → count bullish vs bearish votes from MACD, RSI, MA:
  - 3 bullish → **Strong Buy (+2)**
  - 2 bullish → **Buy (+1)**
  - 3 bearish → **Strong Sell (−2)**
  - 2 bearish → **Sell (−1)**
  - mixed → **Hold (0)**

In [ ]:
def recommendation(pe, macd, rsi, ma):
    '''Combine four indicator signals into a single trading recommendation.

    Arguments
    ---------
    pe   : 0 (chaotic) or 1 (orderly)  -- Permutation Entropy filter
    macd : -1, 0, or 1                 -- MACD signal
    rsi  : -1, 0, or 1                 -- RSI signal
    ma   : -1, 0, or 1                 -- MA Crossover signal

    Returns
    -------
     2  Strong Buy
     1  Buy
     0  Hold / No trade
    -1  Sell
    -2  Strong Sell
    '''
    if pe == 0:
        return 0  # chaotic market: step aside

    bullish = sum(1 for s in [macd, rsi, ma] if s ==  1)
    bearish = sum(1 for s in [macd, rsi, ma] if s == -1)

    if   bullish == 3: return  2
    elif bullish == 2: return  1
    elif bearish == 3: return -2
    elif bearish == 2: return -1
    return 0

In [ ]:
def execute_trades(data_dict, tickers, index_ticker='SPY'):
    '''Run the full indicator pipeline on each trading day for each ticker.
    Opens a position when the recommendation turns non-zero.
    Closes (and optionally reverses) when the direction changes.
    Stores one log-return per closed trade -- only where a trade occurs.

    Returns
    -------
    trade_log_returns        : list[float]
    trade_dates              : list[(entry_date, exit_date, ticker, direction)]
    market_returns_at_trades : list[float]  -- index return on each exit day
    '''
    trade_log_returns        = []
    trade_dates              = []
    market_returns_at_trades = []
    spy_ret = {r[1]: r[3] for r in data_dict.get(index_ticker, [])}

    for ticker in tickers:
        rows   = data_dict[ticker]
        prices = [r[2] for r in rows]
        dates  = [r[1] for r in rows]

        position    = 0      # 0 = flat, 1 = long, -1 = short
        entry_price = None
        entry_date  = None

        for i in range(60, len(prices)):   # 60-bar warmup
            pe  = indicator_1_pe(prices, i)
            mac = indicator_2_macd(prices, i)
            rsi = indicator_3_rsi(prices, i)
            ma  = indicator_4_ma_crossover(prices, i)
            rec = recommendation(pe, mac, rsi, ma)
            desired = 1 if rec >= 1 else (-1 if rec <= -1 else 0)

            # Close existing position when direction reverses or signal goes flat
            if position != 0 and desired != position:
                log_ret = math.log(prices[i] / entry_price)
                if position == -1:
                    log_ret = -log_ret
                trade_log_returns.append(log_ret)
                direction = 'long' if position == 1 else 'short'
                trade_dates.append((entry_date, dates[i], ticker, direction))
                market_returns_at_trades.append(spy_ret.get(dates[i], 0.0))
                position    = 0
                entry_price = None

            # Open new position
            if position == 0 and desired != 0:
                position    = desired
                entry_price = prices[i]
                entry_date  = dates[i]

    return trade_log_returns, trade_dates, market_returns_at_trades

In [ ]:
def compute_performance(trade_log_returns, trade_dates, market_returns_at_trades,
                        rf_annual=0.05):
    '''Compute and print all required performance metrics.

    a. Number of trades per month
    b. Average return + t-test for statistical significance
    c. Average return for long trades
    d. Average return for short trades
    e. Cumulative return, annualised
    f. Sharpe Ratio, annualised
    g. Sortino Ratio, annualised
    h. Jensen\'s Alpha (beta computed using only market returns at trade closes)
    i. VaR at the 5% level
    '''
    if not trade_log_returns:
        print('No trades executed.')
        return {}

    rets    = np.array(trade_log_returns)
    mkt     = np.array(market_returns_at_trades)
    n       = len(rets)
    long_r  = [r for r, (_, _, _, d) in zip(trade_log_returns, trade_dates) if d == 'long']
    short_r = [r for r, (_, _, _, d) in zip(trade_log_returns, trade_dates) if d == 'short']

    # a. Trades per month
    months = {}
    for _, exit_d, _, _ in trade_dates:
        months[exit_d[:7]] = months.get(exit_d[:7], 0) + 1
    avg_tpm = n / max(len(months), 1)

    # b. Average return + t-test
    mean_ret      = float(np.mean(rets))
    t_stat, p_val = stats.ttest_1samp(rets, 0)

    # c/d. Long / short averages
    avg_long  = float(np.mean(long_r))  if long_r  else float('nan')
    avg_short = float(np.mean(short_r)) if short_r else float('nan')

    # e. Cumulative & annualised return
    cum_ret = float(np.sum(rets))
    d0      = datetime.strptime(trade_dates[0][0],  '%Y-%m-%d')
    d1      = datetime.strptime(trade_dates[-1][1], '%Y-%m-%d')
    years   = max((d1 - d0).days / 365.25, 1e-6)
    ann_ret = cum_ret / years

    # f. Sharpe ratio (annualised)
    rf_d   = rf_annual / 252
    excess = rets - rf_d
    sharpe = float(np.mean(excess) / np.std(excess, ddof=1) * np.sqrt(252)) \
             if np.std(excess) > 0 else 0.0

    # g. Sortino ratio (annualised)
    down    = rets[rets < 0]
    d_std   = float(np.std(down, ddof=1)) if len(down) > 1 else 1e-9
    sortino = float((mean_ret / d_std) * np.sqrt(252))

    # h. Jensen's Alpha -- only market returns when a trade is closed
    if len(mkt) > 1 and np.var(mkt) > 0:
        cov   = np.cov(rets, mkt)
        beta  = cov[0, 1] / np.var(mkt, ddof=1)
        alpha = (mean_ret - (rf_d + beta * (np.mean(mkt) - rf_d))) * 252
    else:
        beta  = float('nan')
        alpha = float('nan')

    # i. VaR 5%
    var5 = float(np.percentile(rets, 5))

    # ---- Print report -------------------------------------------------------
    SEP = '=' * 56
    sig = '***' if p_val < 0.01 else ('**' if p_val < 0.05 else ('*' if p_val < 0.10 else 'n.s.'))
    print(SEP)
    print('         STRATEGY PERFORMANCE REPORT')
    print(SEP)
    print('Total Trades           : ' + str(n) +
          '  (Long: ' + str(len(long_r)) + ', Short: ' + str(len(short_r)) + ')')
    print('')
    print('a. Avg Trades / Month  : ' + str(round(avg_tpm, 2)))
    print('')
    print('b. Mean Log-Return     : ' + str(round(mean_ret, 6)))
    print('   t-stat              : ' + str(round(float(t_stat), 4)))
    print('   p-value             : ' + str(round(float(p_val), 4)) + '  (' + sig + ')')
    print('')
    print('c. Avg Return (Longs)  : ' + str(round(avg_long,  6)))
    print('d. Avg Return (Shorts) : ' + str(round(avg_short, 6)))
    print('')
    print('e. Cumulative Return   : ' + str(round(cum_ret * 100, 2)) + '%')
    print('   Annualised Return   : ' + str(round(ann_ret * 100, 2)) + '%')
    print('')
    print('f. Sharpe Ratio        : ' + str(round(sharpe,  4)))
    print('')
    print('g. Sortino Ratio       : ' + str(round(sortino, 4)))
    print('')
    print('h. Beta                : ' + str(round(float(beta),  4)))
    print('   Jensens Alpha       : ' + str(round(float(alpha), 6)))
    print('')
    print('i. VaR (5%)            : ' + str(round(var5 * 100, 2)) + '%')
    print(SEP)

    return dict(n_trades=n, avg_tpm=avg_tpm, mean_return=mean_ret,
                t_stat=float(t_stat), p_value=float(p_val),
                avg_long=avg_long, avg_short=avg_short,
                cum_return=cum_ret, ann_return=ann_ret,
                sharpe=sharpe, sortino=sortino,
                beta=float(beta), jensens_alpha=float(alpha), var5=var5)

---
## Results

In [ ]:
# Run full strategy on all tickers
trade_returns, trade_info, mkt_rets = execute_trades(data, TICKERS, INDEX)
perf = compute_performance(trade_returns, trade_info, mkt_rets)

In [ ]:
def walk_forward_backtest(data_dict, tickers, index_ticker='SPY',
                          train_days=252, test_days=126):
    '''Walk-forward out-of-sample validation.
    Splits the data into rolling train / test windows.
    Parameters are fixed (no re-fitting); this tests stability across periods.

    train_days : in-sample window length (default 252 = 1 year)
    test_days  : out-of-sample window length (default 126 = ~6 months)
    '''
    print('=' * 60)
    print('           WALK-FORWARD BACKTEST RESULTS')
    print('  Train: ' + str(train_days) + ' days  |  Test: ' + str(test_days) + ' days')
    print('-' * 60)

    n_dates = min(len(v) for v in data_dict.values())
    folds   = []
    start   = train_days
    while start + test_days <= n_dates:
        folds.append((start, start + test_days))
        start += test_days

    print('  Total folds: ' + str(len(folds)))
    print('-' * 60)
    print('  Fold   Test Start     Test End    Ann.Ret%   Sharpe   Win%')
    print('-' * 60)

    all_oos = []
    for k, (ts, te) in enumerate(folds):
        fold_data = {t: data_dict[t][ts:te] for t in tickers + [index_ticker]}
        tr, td, mr = execute_trades(fold_data, tickers, index_ticker)
        if not tr:
            print('   ' + str(k + 1) + '   (no trades in this fold)')
            continue

        all_oos.extend(tr)
        r   = np.array(tr)
        d0  = datetime.strptime(td[0][0],  '%Y-%m-%d')
        d1  = datetime.strptime(td[-1][1], '%Y-%m-%d')
        yrs = max((d1 - d0).days / 365.25, 1e-6)
        ann = float(np.sum(r)) / yrs
        exc = r - 0.05 / 252
        sh  = float(np.mean(exc) / np.std(exc, ddof=1) * np.sqrt(252)) \
              if np.std(exc) > 0 else 0.0
        win = float(np.mean(r > 0)) * 100
        s   = fold_data[tickers[0]][0][1]  if fold_data[tickers[0]] else '---'
        e   = fold_data[tickers[0]][-1][1] if fold_data[tickers[0]] else '---'
        line = ('  ' + str(k + 1).rjust(4) + '   ' + s + '   ' + e +
                '   ' + str(round(ann * 100, 2)).rjust(8) + '%' +
                '   ' + str(round(sh, 3)).rjust(6) +
                '   ' + str(round(win, 1)).rjust(5) + '%')
        print(line)

    if all_oos:
        r_all  = np.array(all_oos)
        t, p   = stats.ttest_1samp(r_all, 0)
        print('-' * 60)
        print('  OOS avg return : ' + str(round(float(np.mean(r_all)), 6)) +
              '   t=' + str(round(float(t), 3)) +
              '   p=' + str(round(float(p), 4)))
    print('=' * 60)


walk_forward_backtest(data, TICKERS, INDEX)

In [ ]:
if trade_returns:
    cum_curve  = np.cumsum(trade_returns)
    exit_dates = [td[1] for td in trade_info]
    dt_exits   = [datetime.strptime(d, '%Y-%m-%d') for d in exit_dates]

    fig, ax = plt.subplots(figsize=(12, 5))
    ax.plot(dt_exits, cum_curve, linewidth=1.5, color='steelblue',
            label='Strategy cumulative log-return')
    ax.axhline(0, color='black', linewidth=0.6, linestyle='--')
    ax.set_title('Cumulative Log-Return -- Permutation Entropy Strategy', fontsize=13)
    ax.set_xlabel('Trade Exit Date')
    ax.set_ylabel('Cumulative Log-Return')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print('No trades to plot.')